# ML12 Future Topics: Code Illustrations

This notebook illustrates the main ideas from `index.md`:

- Hyperparameters and tuning
- Pipelines and leakage prevention
- Ensemble methods (bagging vs boosting)
- Random Forest and Gradient Boosting behavior
- Interpretability vs accuracy tradeoff
- Bayesian updating and uncertainty

In [ ]:
# Core libraries for numerical computing, data handling, and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-learn tools for data generation, model selection, pipelines, and model evaluation
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

np.random.seed(42)
# Set seed for reproducibility

## 1) Build a Synthetic Classification Problem

We create a tabular dataset with nonlinear signal and noise. This makes the differences between models visible.

In [ ]:
# Generate a synthetic binary classification dataset
# 3000 samples, 20 features (8 informative, 4 redundant, 8 noise)
# class_sep controls separation between classes; flip_y adds label noise
X, y = make_classification(
    n_samples=3000,
    n_features=20,
    n_informative=8,
    n_redundant=4,
    class_sep=1.0,
    flip_y=0.03,
    random_state=42,
)

# Create a DataFrame with named features for easier interpretation
feature_names = [f"x{i}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_names)

# Inject missing values (~3%) to simulate real-world incomplete data
# This demonstrates why pipelines with imputation are important
mask = np.random.rand(*df.shape) < 0.03
df = df.mask(mask)


# Split data into train/test (75/25) with stratified sampling to preserve class balanceX_train.shape, X_test.shape
X_train, X_test, y_train, y_test = train_test_split(# Display the dimensions of the training and test sets
    df, y, test_size=0.25, random_state=42, stratify=y
)

((2250, 20), (750, 20))

## 2) Hyperparameters: Why Tuning Matters

Hyperparameters control the learning strategy. Here we tune a Random Forest over `n_estimators` and `max_depth`

In [15]:
# Initialize a Random Forest classifier
rf = RandomForestClassifier(random_state=42)

# Define hyperparameter grid for tuning:
# - n_estimators: number of trees (trade-off between performance and speed)
# - max_depth: tree depth (controls overfitting)
param_grid = {
    "n_estimators": [50, 100, 150],
    "max_depth": [None, 5, 8],
}

# Use GridSearchCV to exhaustively search parameter combinations
grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="roc_auc",
)

# Explain AUC and CV AUC
print("AUC (Area Under the ROC Curve) measures the ability of the model to distinguish between classes. Higher is better (max 1.0).")
print("CV AUC (Cross-Validated Area Under the ROC Curve) is the average AUC across all folds in cross-validation.")

# Fit the grid to find the best hyperparameters
grid.fit(X_train, y_train)
print("Test AUC (best RF):", round(test_auc, 4))
print("Best CV AUC:", round(grid.best_score_, 4))

# Extract the best-performing model and evaluate on test set
print("Best params:", grid.best_params_)
best_rf = grid.best_estimator_ # Print results: best hyperparameters and corresponding performance

test_auc = roc_auc_score(y_test, best_rf.predict_proba(X_test)[:, 1])

AUC (Area Under the ROC Curve) measures the ability of the model to distinguish between classes. Higher is better (max 1.0).
CV AUC (Cross-Validated Area Under the ROC Curve) is the average AUC across all folds in cross-validation.
Test AUC (best RF): 0.9493
Best CV AUC: 0.9299
Best params: {'max_depth': None, 'n_estimators': 150}


## 3) Pipelines: Reproducible End-to-End Workflow

A pipeline ensures imputation + scaling + model fitting happen together inside cross-validation, reducing leakage risk.

In [17]:
# Manual (non-pipeline) 
# 
# 1. Fit imputer on training data and transform train/test
# 2. Fit scaler on imputed training data and transform train/test
# 3. Train logistic regression
# 4. Evaluate test AUC


# 1) Impute missing values using only training data statistics
imputer_manual = SimpleImputer(strategy="median")
X_train_imputed = pd.DataFrame(
    imputer_manual.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
)
X_test_imputed = pd.DataFrame(
    imputer_manual.transform(X_test),
    columns=X_test.columns,
    index=X_test.index,
)

# 2) Scale using only training data fit
scaler_manual = StandardScaler()
X_train_scaled = scaler_manual.fit_transform(X_train_imputed)
X_test_scaled = scaler_manual.transform(X_test_imputed)

# 3) Train logistic regression on transformed training data
logit_manual = LogisticRegression(max_iter=2000)
logit_manual.fit(X_train_scaled, y_train)

# 4) Evaluate on transformed test data
manual_test_auc = roc_auc_score(y_test, logit_manual.predict_proba(X_test_scaled)[:, 1])

print("Manual workflow test AUC:", round(manual_test_auc, 4))
print("Pipeline workflow test AUC:", round(test_auc_pipe, 4))

Manual workflow test AUC: 0.8484
Pipeline workflow test AUC: 0.8484


In [18]:
# Identify numeric feature columns for preprocessing
numeric_features = list(X_train.columns)

# Build preprocessing pipeline: imputation → standardization
# ColumnTransformer applies these transformations to all numeric columns
preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                # Impute missing values with median (robust to outliers)
                ("imputer", SimpleImputer(strategy="median")),
                # Standardize features to have mean=0, std=1 (required for logistic regression)
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        )
    ]
)

# Complete end-to-end pipeline: preprocessing → logistic regression
# Ensures preprocessing happens INSIDE cross-validation (prevents data leakage)
pipe_logit = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=2000)),
    ]
)


# Evaluate the pipeline with 5-fold cross-validation (AUC metric)
cv_auc = cross_val_score(pipe_logit, X_train, y_train, cv=5, scoring="roc_auc")# Print results: cross-validation and test performance
print("Pipeline CV AUC (mean):", round(cv_auc.mean(), 4))

# Fit the pipeline on the entire training set
pipe_logit.fit(X_train, y_train)

# Evaluate on test set
test_auc_pipe = roc_auc_score(y_test, pipe_logit.predict_proba(X_test)[:, 1])
print("Pipeline test AUC:", round(test_auc_pipe, 4))


Pipeline CV AUC (mean): 0.8209
Pipeline test AUC: 0.8484


## 4) Ensemble Comparison: Single Tree vs Random Forest vs Boosting

- Decision Tree: interpretable but high variance
- Random Forest: bagging reduces variance
- Gradient Boosting: sequential error correction can reduce bias

In [ ]:
# Define three ensemble models to compare:
#
# 1. Decision Tree: single tree (high variance, prone to overfitting)
#
# 2. Random Forest: ensemble by bagging (reduces variance)
#    A random forest is different from a decision tree because it builds multiple trees on random subsets of the data and features, 
#    then averages their predictions to improve generalization and reduce overfitting.
#
# 3. Gradient Boosting: sequential ensemble (reduces bias). This model is more complex and often achieves better performance but can be slower to train.
#    It works by fitting trees sequentially, where each new tree tries to correct the errors of the previous ones. This can lead to better performance, 
#    especially on complex datasets, but it can also be more sensitive to hyperparameters and may require more careful tuning.
#
# Variance is the amount by which the model's predictions would change if we used a different training dataset. 
# High variance models (like decision trees) can fit the training data very closely but may not generalize well to new data (overfitting).
#
# Bias is the error introduced by approximating a real-world problem (which may be complex) with a simpler model. High bias models 
# (like linear regression) may underfit the data, failing to capture important patterns.

models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

# Train each model and collect performance metrics
rows = []
for name, model in models.items():
    # Fit on training data (fillna for missing values)
    model.fit(X_train.fillna(X_train.median()), y_train)
    # Get predicted probabilities on test set
    probs = model.predict_proba(X_test.fillna(X_train.median()))[:, 1]
    # Convert probabilities to binary predictions (threshold=0.5)
    preds = (probs >= 0.5).astype(int)
    # Record both accuracy and AUC for this model
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, preds),
        "auc": roc_auc_score(y_test, probs),
    })

# Combine results into a DataFrame and sort by AUC (best to worst)
results = pd.DataFrame(rows).sort_values("auc", ascending=False)
results

,model,accuracy,auc
1,Random Forest,0.878667,0.951856
2,Gradient Boosting,0.882667,0.948962
0,Decision Tree,0.772000,0.771915


## 6) OLS vs Bayesian Linear Regression (Same Problem)

This section uses one simple linear regression problem and solves it two ways:

1. **Frequentist OLS**: estimate a single best-fit coefficient and intercept.
2. **Bayesian linear regression**: estimate a posterior distribution over coefficients.

The point is to show how both approaches can produce similar central estimates, while the Bayesian version naturally reports uncertainty as distributions.

In [27]:
# Build a simple 1-feature regression dataset
rng = np.random.default_rng(42)
n = 120
x = np.linspace(0, 10, n)
true_intercept = 3.0
true_slope = 2.5
sigma = 2.0

# y = beta0 + beta1*x + noise
y_reg = true_intercept + true_slope * x + rng.normal(0, sigma, size=n)

# ------------------------------------------------------------------
# 1) Frequentist OLS (point estimate) using statsmodels
# ------------------------------------------------------------------
import statsmodels.api as sm
from sklearn.linear_model import BayesianRidge

# statsmodels expects us to explicitly add the intercept column
X_ols = sm.add_constant(x)
ols_model = sm.OLS(y_reg, X_ols).fit()

beta_ols = ols_model.params  # [intercept, slope]
yhat_ols = ols_model.predict(X_ols)
sigma2_hat = np.mean(ols_model.resid**2)

print("OLS estimates (statsmodels):")
print(f"  intercept: {beta_ols[0]:.3f}")
print(f"  slope:     {beta_ols[1]:.3f}")

# ------------------------------------------------------------------
# 2) Bayesian linear regression using BayesianRidge
# ------------------------------------------------------------------
# Use the same design matrix with an explicit intercept term so both
# intercept and slope are treated as Bayesian coefficients.
X_bayes = sm.add_constant(x)
bayes_model = BayesianRidge(fit_intercept=False)
bayes_model.fit(X_bayes, y_reg)

mN = bayes_model.coef_  # posterior mean of [intercept, slope]
SN = bayes_model.sigma_  # posterior covariance of coefficients
post_sd = np.sqrt(np.diag(SN))
ci_low = mN - 1.96 * post_sd
ci_high = mN + 1.96 * post_sd

print("\nBayesian posterior (BayesianRidge, approx. 95% credible intervals):")
print(f"  intercept mean: {mN[0]:.3f}  CI: [{ci_low[0]:.3f}, {ci_high[0]:.3f}]")
print(f"  slope mean:     {mN[1]:.3f}  CI: [{ci_low[1]:.3f}, {ci_high[1]:.3f}]")

# Numeric comparison table only (visual plot removed)
comparison = pd.DataFrame({
    "parameter": ["intercept", "slope"],
    "OLS_point_estimate": beta_ols,
    "Bayesian_posterior_mean": mN,
    "Bayesian_95pct_CI_low": ci_low,
    "Bayesian_95pct_CI_high": ci_high,
})
comparison

OLS estimates (statsmodels):
  intercept: 3.217
  slope:     2.432

Bayesian posterior (BayesianRidge, approx. 95% credible intervals):
  intercept mean: 3.189  CI: [2.636, 3.743]
  slope mean:     2.436  CI: [2.341, 2.532]


,parameter,OLS_point_estimate,Bayesian_posterior_mean,Bayesian_95pct_CI_low,Bayesian_95pct_CI_high
0,intercept,3.217359,3.189258,2.635923,3.742592
1,slope,2.432367,2.436382,2.340625,2.532138


### Interpreting OLS vs Bayesian Intervals

Using the results above:

- **OLS output** gives a single point estimate for each parameter.
- **Bayesian output** gives a posterior distribution and a credible interval for each parameter.

Interpretation difference:

- **Frequentist confidence interval (95%)**: across many repeated samples, 95% of intervals built this way would contain the true parameter.
- **Bayesian credible interval (95%)**: given the data and prior, there is 95% posterior probability that the parameter lies in this interval.

So both can look numerically similar, but they answer different probability questions.